# <center>Si ustedes fueran a invertir en un Airbnb en alguna ciudad,</center>  
# <center> ¿qué información necesitarían antes de decidir?</center>

## Análisis de hosts y sus propiedades


In [ ]:
# Cargar los archivos CSV en DataFrames
import pandas as pd 
airbnb = pd.read_csv('../Cambridge.csv')

In [ ]:
airbnb.head()

¿qué tipo de alojamiento predomina?

¿Cuál es el tipo de alojamiento más común en Cambridge?

"Ya sabemos qué tipo de alojamiento aparece más. Pero ahora quiero saber cuánto cuesta, en promedio, cada tipo."

In [ ]:
airbnb.groupby("room_type")["price"].mean()

In [ ]:
airbnb["price"].dtype
airbnb["price"].head()

In [ ]:
airbnb["price"] = (
    airbnb["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

¿Qué tipo de alojamiento tiene el precio promedio más alto?

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.histplot(data=airbnb, x="price")
plt.show()

¿Qué observan?



In [ ]:
sns.histplot(data=airbnb, x="price", bins=30, kde=True)
plt.show()

Si cambio bins, ¿cambiaron los datos?


## Ejercicio 1: ¿Cuántas propiedades tiene cada host?
Usa groupby() y merge() para unir la información de hosts y calcular cuántas propiedades administra cada uno.

In [ ]:
hosts = airbnb[['host_id', 'host_name', 'host_since', 'host_is_superhost', 'host_listings_count']]
listings = airbnb[['host_id', 'id', 'name', 'property_type', 'room_type', 'price']]
print(hosts)
print(listings)





In [ ]:
# Contar cuántas propiedades tiene cada host
df_host_count = df_hosts.groupby(['host_id', 'host_name', 'host_is_superhost'])['id'].count().reset_index()
print(f"Número de propiedades por host: \n{df_host_count}")
df_host_count.rename(columns={'id': 'num_listings'}, inplace=True)

print(f"Número de propiedades por host: \n")
df_host_count.sort_values(
    "num_listings",
    ascending=False
).head(10)

¿Quien administra más propiedades?

## ¿Cuáles son los superhosts más activos?
Filtra solo los superhosts (host_is_superhost == 't') y ordena por número de propiedades.

In [ ]:
superhosts = df_host_count[df_host_count['host_is_superhost'] == 't']
print(f"Superhosts: \n{superhosts}")
superhosts = superhosts.sort_values(by='num_listings', ascending=False)
print("Los 10 superhosts más activos: \n ")  # Los 10 superhosts más 
superhosts.head(10)


¿Los hosts con más propiedades son superhosts?

## Análisis de precios y tipos de propiedades
Ahora combinaremos datos de precios, tipos de propiedades y ubicación para descubrir tendencias de precios.

¿Cuál es el precio promedio por tipo de propiedad?

 ¿Qué barrios son más caros?

¿Por qué podría ser útil conocer la mediana y no solamente el promedio?  
La diferencia entre media y mediana (valores extremos o outliers). La media se "arrastra" por valores extremos. Si en un barrio hay una mansión de $10,000 entre 9 propiedades normales de $200, la media subirá artificialmente, aunque la mayoría de alojamientos sean baratos. La mediana —el valor central que deja la mitad por debajo y la mitad por arriba— no se ve afectada por ese extremo. Comparar ambas revela si un barrio es realmente caro o si solo tiene unos pocos alojamientos de lujo.

In [ ]:
sns.histplot(data=df_price_neigh, x="mean")
plt.show()

In [ ]:
sns.histplot(data=df_price_neigh, x="median")
plt.show()

## Agregación y agrupamiento de datos
Objetivo:
Aplicar funciones de agregación (groupby(), agg()) para encontrar patrones en los datos.

Analizar disponibilidad, ocupación y precios de hospedaje en cada ciudad.

Aplicar pivot_table() para estructurar la información de manera efectiva.

## Análisis de disponibilidad y ocupación
Los estudiantes explorarán la disponibilidad de propiedades en su ciudad, detectando cuáles son las zonas con mayor oferta y ocupación.

¿Cuáles son los barrios con más alojamientos disponibles en los próximos 30 días?

In [ ]:
# Cargar los archivos CSV en DataFrames
import pandas as pd 

df_avail = (
    airbnb
    .groupby("neighbourhood_cleansed")
    .agg(
        disponibilidad_total=("availability_30", "sum"),
        alojamientos=("id", "count")
    )
    .sort_values(
        "disponibilidad_total",
        ascending=False
    )
)

df_avail.head(10)


sumar availability_30 mezcla dos cosas distintas. Un barrio puede tener mucha disponibilidad total simplemente porque tiene muchos alojamientos, no porque cada uno esté más disponible. Por eso se crea la segunda métrica: el promedio normaliza por barrio (disponibilidad por alojamiento).

Esto lleva a distinguir dos preguntas que dan respuestas diferentes:

Barrio con más disponibilidad (total). Favorece a barrios grandes, con muchos listados. Es una medida de volumen.

Barrio con mayor disponibilidad promedio por alojamiento. Favorece a barrios donde cada alojamiento individual tiene más noches libres. Es una medida de intensidad o de cuán "desocupado" está cada lugar.

No necesariamente coinciden: un barrio pequeño cuyos pocos alojamientos están casi siempre libres podría tener alta disponibilidad promedio pero baja disponibilidad total, y viceversa.


Porcentaje de alojamientos ocupados en los próximos 30 días

In [ ]:
porcentaje = (airbnb['availability_30'] == 0).mean() * 100
print(f"📌 {porcentaje:.2f}% de los alojamientos están completamente ocupados.")


In [ ]:
airbnb['availability_30'].eq(0).value_counts(normalize=True) * 100 # Porcentaje de alojamientos completamente ocupados

# Aquí se calcula el porcentaje de alojamientos completamente ocupados en los próximos 30 días
#  utilizando la función `eq()` para comparar la columna `availability_30` con 0, 
# y luego se utiliza `value_counts(normalize=True)` para obtener la proporción de True y False, multiplicando por 100 para obtener el porcentaje.

## Comparación de precios y reseñas

Se analizarán tendencias de calificaciones y precios para entender qué factores impactan en la valoración de una propiedad.

Ejercicio 3: ¿Los alojamientos más caros tienen mejores reseñas?

In [ ]:
sns.scatterplot(
    data=airbnb,
    x="price",
    y="review_scores_rating",
    alpha=0.5 # Ajusta la transparencia de los puntos para mejorar la visibilidad
)

plt.show()

¿Qué pasaría si quitamos temporalmente los precios extremadamente altos?

In [ ]:
airbnb_grafica = airbnb[
    (airbnb["price"] <= 1000) &
    (airbnb["review_scores_rating"].notna())
]
sns.scatterplot(
    data=airbnb_grafica,
    x="price",
    y="review_scores_rating",
    alpha=0.5
)

plt.show()

 Relación entre número de reseñas y calificación promedio

In [ ]:
df_review_count = airbnb.groupby('number_of_reviews')['review_scores_rating'].mean().reset_index().sort_values(by='number_of_reviews', ascending=False)

print(df_review_count.head(20))  # Mostrar los 20 valores con más reseñas
